# Tools

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

GOOGLE_API_KEY = "GOOGLE_API_KEY"
GROQ_API_KEY = "GROQ_API_KEY"
OPENAI_API_KEY = "OPENAI_API_KEY"

def require_env(name: str) -> str:
    value = os.getenv(name)
    if value is None:
        raise ValueError(f"{name} is not set. Add it to your .env file.")
    os.environ[name] = value
    return value

google_api_key = require_env(GOOGLE_API_KEY)
groq_api_key = require_env(GROQ_API_KEY)
openai_api_key = require_env(OPENAI_API_KEY)

In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:qwen/qwen3.8-27b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots don’t actually "talk" in the human sense. Instead, they **mimic** sounds, including human speech, for several biological and social reasons. Here’s why:\n\n### 1. **Social Bonding and Communication**\nParrots are highly social, flock-oriented birds. In the wild, they use a wide range of vocalizations to:\n- Stay in contact with flock members.\n- Warn of predators.\n- Reinforce social bonds.\n- Express emotions (e.g., excitement, distress, joy).\n\nWhen kept as pets, they view their human caregivers as part of their "flock." Mimicking human speech is a way to **integrate into their social group** and maintain that bond.\n\n### 2. **Vocal Learning Ability**\nParrots possess a unique neurological and anatomical ability to learn and reproduce complex sounds. This skill is called **vocal learning**, which is rare in the animal kingdom (also seen in songbirds, dolphins, and elephants).\n- Their syrinx (voice box) is highly flexible, allowing precise control over pi

In [4]:
from langchain.tools import tool

@tool
def get_weather(location:str) -> str:
    """
    Get weather at a specific location
    :param location:
    :return:
    """
    return f"It's sunny in {location}"

# bild tool to model
model_with_tools = model.bind_tools([get_weather])

In [5]:
response = model_with_tools.invoke("What's the weather in Weatherland?")

print(response)

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': 'q7qtcz4fr', 'function': {'arguments': '{"location":"Weatherland"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 284, 'total_tokens': 311, 'completion_time': 0.075091564, 'completion_tokens_details': None, 'prompt_time': 0.025290451, 'prompt_tokens_details': None, 'queue_time': 0.05261866, 'total_time': 0.100382015}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_43dd1ea4ee', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0e88a-adf7-71a3-be82-67983f91aaea-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Weatherland'}, 'id': 'q7qtcz4fr', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 284, 'output_tokens': 27, 'total_tokens': 311}
Tool: get_weather
Args: {'location': 'Weatherland'}


### Tool Execution Loops

In [7]:
# Step 1: Model generates tool calls
messages = [{
    "role": "user",
    "content": "What's the weather in WeatherLand?"
}]

ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

It's sunny in WeatherLand! ☀️


In [8]:
messages

[{'role': 'user', 'content': "What's the weather in WeatherLand?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '1hqb8ad9d', 'function': {'arguments': '{"location":"WeatherLand"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 284, 'total_tokens': 311, 'completion_time': 0.074094112, 'completion_tokens_details': None, 'prompt_time': 0.02334821, 'prompt_tokens_details': None, 'queue_time': 0.089235983, 'total_time': 0.097442322}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_4560dae850', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0e88c-8aba-7c62-ba0e-19fec80628e5-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'WeatherLand'}, 'id': '1hqb8ad9d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 284, 'output_tokens': 27, 'total_tokens': 311}),
 ToolMessage(conten